Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:


Tracking agent behavior with logging, analytics, and debugging.

Transforming prompts, tool selection, and output formatting.

Adding retries, fallbacks, and early termination logic.

Applying rate limits, guardrails, and PII detection.

Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.
Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [5]:
### Messagebased summarization
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [6]:
config={"configurable":{"thread_id":"test-1"}}

In [7]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='fa09d1ec-671e-4b0c-bdb9-a8e87f1244e4'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019feb19-67bb-7912-a770-814224ae4c06-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 28, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='fa09d1ec-671e-4b0c-bdb9-a8e87f1244e4'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019feb19-67bb-7912-a770-814224ae4c06-0', tool_c

## token Size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

In [11]:
### Messagebased summarization
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

In [13]:
config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars//4



In [14]:
cities=["Paris","London","Tokyo","New York","Dubai","Singapoore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config

    )
    tokens =count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~96 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='ecceeac4-08a1-422d-9dcf-eb15b53a2a04'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'5963be30-251e-473c-a54a-0bb4becb3299': 'CpQCARFNMg8s68jiIvzLPC9kLs2l5NdU4nuQigiNEkdhrTTJ1F5tytOVlYbDR9cI7ylnUeVTPxAB1qogz3d+FXOLNYb6MY+vKu+QwzIxj/MAFwVVItpVTRMk1OYYMNgeVbDZi4BE4MCe+MF9B0gLSt7srs1rzzZMpBQHGLd1oyoCos9tVHGXiXMerpKAs+oZYJ/3lBV+C8Z46OM5wjDSGfgmYM68lCx9bPR/LZCj7duJlwBeDaN08B61MdOT4aS+4KHjx0VyaLdYYkJhoSjrwLc+vGpUQf/lRvw5zkg1GMgCWCZUS6/xF3dGXo3UhQQdfWAAVnmj7i6ENm6B3HdNV3+TRfDhoJD1OHXB//ycqTw5lglddaLi'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019feb35-dd92-7463-aee6-3df6d662813e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}

# Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).
Compliance workflows where human oversight is mandatory.
Long-running conversations where human feedback guides the agent.

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [21]:
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
       HumanInTheLoopMiddleware(
           interrupt_on={
               "send_email_tool":{
                   "allowed_decisions":["approve","edit","reject"]
               },
               "read_email_tool":False
           }
       )
    ]
)

In [ ]:
config={"configurable":{"thread_id":"test-1"}}

In [23]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to yogesh@gmail.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to yogesh@gmail.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b52f881f-b647-4774-b9bc-4077491c49c5'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "yogesh@gmail.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'d42d1c88-39fb-4159-ac5a-23afff4a7775': 'CoIFARFNMg+2+F1kqg8Ryx+PHICZ33/UwGt2K4eILf3UcELwNWMtf60TF4yk2rn0YGhXCqxb+eSaDsR1p849XvSzsv4ddW+UHL0PLGf0MImdwoUwR0BM7yKdzxK3xSbWg4y5/92CN/c2t9QN+48ntoGesowiDD3ehnHXpP2i25LdNmPiPlokV4+F5RuvzMNv2R0FQjvyoiDFWkGzzUsWyWSqg8Ex3Fyfjq0VuakzW1bnAExlX6OIqfNuPpyuszei4IYR/svNEeCCkaT1K1v4OEoRC8Ms+Ivg2aqciZ7YjUs8I27h4LscnjW+QQe2AYXvbYLjVBCyF/DcXR5hdMOGmNh1qnw2Pzm21yVpEBL9FTeZ1GUpUHoKpQgQWUE0mBxiV4YHdSD2ki11PEHc/YBWiuST8395sB8Y/qSL/gG9L+weck/Ph9xhtSSUXDhJAycxl/gUBh+F8QQs7mdjw+2ZBTF54asw7Dm3h1QHLH6CzV1kbS3lS+MUSS99G63RXLbyNKQfSegPNTgFm0

In [25]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I have sent the email to yogesh@gmail.com with the subject 'Hello' and body 'How are you?'.


## Reject

In [26]:
agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
       HumanInTheLoopMiddleware(
           interrupt_on={
               "send_email_tool":{
                   "allowed_decisions":["approve","edit","reject"]
               },
               "read_email_tool":False
           }
       )
    ]
)

config={"configurable":{"thread_id":"test-1"}}


result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to yogesh@gmail.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)


from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I can't send the email. It seems the tool call was rejected.


In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
       HumanInTheLoopMiddleware(
           interrupt_on={
               "send_email_tool":{
                   "allowed_decisions":["approve","edit","reject"]
               },
               "read_email_tool":False
           }
       )
    ]
)

In [28]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='f94bde76-51be-4d1e-8520-3acd773c4774'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Test", "body": "Hello", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'df51bc8a-505e-4d56-862d-7db903fa97fb': 'CrIEARFNMg9Y+0r+hEKPhWoalMU7A4+qk9u0jMDEumvV+h7pMVz3+9MQT4wYA8iPPQtH7pSbXoMKs/tq8vQ0lK5jM6/cuKOmmFua7zaDg7YdssdQVVD6iVx0Vr3MNM8QM20gLmlXT0LsBaVMgLef/rRlj89aWKfeW31nIGWwcHtUaMR5Iqq1/5Il0zqmxqR7vzWrRp+hv7wZX2ZK/hi65WezYCbq6DMBzY7y2Cfckt3hN5S4nLdDUOT9gEeF/56rCQdqNPnVSCWxU+eQFSjDNZSi/PeroXXwBhQlwaiWNY+dNe8G03r95Z3Lz0BTS/uWbox347PR/rsNNeB6OnbBEb7BuNqN1LkmpAE4513mXM+3NjexcvV+Pm1hWpnqPSsWVYYZiFw7Plw8EPk7iUPWCdhPd08HZPkx0NUZwJUGTbbq7XJYcMi/DvdrxogJlC+EGmNsdE6mEwFHybks1D1yNxR4L3IEfNmXmp+lAuUxyTVxlIK5HWnm2WLPxMBgyCItKPnDbQ0Zv4ofIv0GyVjc0kJrnXdBUM/7

In [29]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: I have sent the email to 'correct@email.com' with the subject 'Corrected Subject' and body 'This was edited by human before sending'.


In [30]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='f94bde76-51be-4d1e-8520-3acd773c4774'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Test", "body": "Hello", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'df51bc8a-505e-4d56-862d-7db903fa97fb': 'CrIEARFNMg9Y+0r+hEKPhWoalMU7A4+qk9u0jMDEumvV+h7pMVz3+9MQT4wYA8iPPQtH7pSbXoMKs/tq8vQ0lK5jM6/cuKOmmFua7zaDg7YdssdQVVD6iVx0Vr3MNM8QM20gLmlXT0LsBaVMgLef/rRlj89aWKfeW31nIGWwcHtUaMR5Iqq1/5Il0zqmxqR7vzWrRp+hv7wZX2ZK/hi65WezYCbq6DMBzY7y2Cfckt3hN5S4nLdDUOT9gEeF/56rCQdqNPnVSCWxU+eQFSjDNZSi/PeroXXwBhQlwaiWNY+dNe8G03r95Z3Lz0BTS/uWbox347PR/rsNNeB6OnbBEb7BuNqN1LkmpAE4513mXM+3NjexcvV+Pm1hWpnqPSsWVYYZiFw7Plw8EPk7iUPWCdhPd08HZPkx0NUZwJUGTbbq7XJYcMi/DvdrxogJlC+EGmNsdE6mEwFHybks1D1yNxR4L3IEfNmXmp+lAuUxyTVxlIK5HWnm2WLPxMBgyCItKPnDbQ0Zv4ofIv0GyVjc0kJrnXdBUM/7